# Stage 05 — Rating Scale Calibration

Calibrate the rating scale to the portfolio target central tendency. Assign rating grades and calibrated PDs. Perform stress testing.

**Key principle:** Calibration uses **model-predicted PDs** per grade (not observed default rates) to avoid circular calibration.

In [ ]:
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Set working directory to project root
os.chdir('c:/projects/superagent')

sys.path.insert(0, 'src')
import pdtoolkit as pdt

RUN_DIR = 'runs/2026-03-15_201852'

# Load binned dataset
df = pd.read_csv(f'{RUN_DIR}/data/loans_binned.csv')
print(f"Loaded binned dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Default rate: {df['Creditability'].mean():.4f}")

# Load model parameters
with open(f'{RUN_DIR}/pipeline/model_params.json', 'r') as f:
    model_params = json.load(f)

selected_vars = model_params['selected_variables']
woe_mappings = model_params['woe_mappings']
coefficients = model_params['coefficients']
intercept = model_params['intercept']
score_params = model_params['score_params']

print(f"\nChampion model: {model_params['selection_method']}")
print(f"Variables ({len(selected_vars)}): {selected_vars}")
print(f"Intercept: {intercept:.4f}")
print(f"Score params: base={score_params['base_score']}, odds={score_params['base_odds']}, PDO={score_params['pdo']}")

## 1. Compute Model-Predicted Probabilities and Scores

For each observation, WoE-encode using the mappings, compute the logit, convert to predicted probability, and then to a scaled score.

In [ ]:
# Build WoE lookup for each variable: bin_label -> woe_value
woe_lookup = {}
for var in selected_vars:
    mapping = woe_mappings[var]
    lookup = {}
    for entry in mapping:
        lookup[entry['bin']] = entry['woe']
    woe_lookup[var] = lookup

# WoE-encode each observation and compute logit
logits = np.full(len(df), intercept)

for var in selected_vars:
    lookup = woe_lookup[var]
    coef = coefficients[var]
    woe_values = df[var].map(lookup).astype(float)
    # Check for unmapped bins
    n_missing = woe_values.isna().sum()
    if n_missing > 0:
        print(f"WARNING: {var} has {n_missing} unmapped bins, filling with 0.0")
        woe_values = woe_values.fillna(0.0)
    logits += coef * woe_values.values

# Convert logit to predicted probability: P(default) = 1 / (1 + exp(-logit))
df['predicted_prob'] = 1.0 / (1.0 + np.exp(-logits))

# Convert predicted probability to scaled score
df['score'] = pdt.scaled_score(
    df['predicted_prob'].values,
    score=score_params['base_score'],
    odd=score_params['base_odds'],
    pdo=score_params['pdo']
)

print(f"Predicted probability — min: {df['predicted_prob'].min():.4f}, "
      f"max: {df['predicted_prob'].max():.4f}, mean: {df['predicted_prob'].mean():.4f}")
print(f"Score — min: {df['score'].min():.1f}, max: {df['score'].max():.1f}, mean: {df['score'].mean():.1f}")

# Target central tendency = development sample default rate
TARGET_CT = df['Creditability'].mean()
print(f"\nTarget central tendency (dev sample DR): {TARGET_CT:.4f}")

## 2. Define Rating Grades

Assign observations to rating grades based on score quantiles (aiming for 8 grades with meaningful boundaries).

In [ ]:
# Define grade boundaries using quantiles for ~8 grades
n_grades = 8
quantiles = np.linspace(0, 1, n_grades + 1)
score_boundaries = np.quantile(df['score'].values, quantiles)

# Round boundaries and ensure uniqueness
score_boundaries = np.round(score_boundaries).astype(int)
score_boundaries = np.unique(score_boundaries)

# If we lost grades due to ties, adjust
if len(score_boundaries) < n_grades + 1:
    # Use rounded percentile-based approach
    score_boundaries = np.percentile(df['score'].values, np.linspace(0, 100, n_grades + 1))
    score_boundaries = np.unique(np.round(score_boundaries, 0))

# Assign grades (higher score = better grade = lower grade number)
# Grade 1 = best (highest scores), Grade N = worst (lowest scores)
df['score_rounded'] = df['score'].values
labels = []
grade_info = []

# Use pd.cut with the boundaries
# Scores are higher=better, so reverse: grade 1 = highest scores
actual_n_grades = len(score_boundaries) - 1
bins = sorted(score_boundaries)
# Extend the edges slightly to capture all values
bins[0] = bins[0] - 1
bins[-1] = bins[-1] + 1

df['grade_num'] = pd.cut(
    df['score'],
    bins=bins,
    labels=list(range(actual_n_grades, 0, -1)),  # Reverse: high score = grade 1
    include_lowest=True
).astype(int)

# Create grade labels
grade_labels = {i: f"Grade {i}" for i in range(1, actual_n_grades + 1)}
df['grade'] = df['grade_num'].map(grade_labels)

print(f"Number of rating grades: {actual_n_grades}")
print(f"\nScore boundaries: {sorted(bins)}")
print(f"\nGrade distribution:")
print(df.groupby('grade_num').agg(
    n=('Creditability', 'count'),
    score_min=('score', 'min'),
    score_max=('score', 'max'),
    predicted_pd=('predicted_prob', 'mean'),
    observed_dr=('Creditability', 'mean')
).sort_index())

## 3. Build Rating Scale and Calibrate

Build the rating scale DataFrame using **model-predicted PDs** (not observed DRs) and calibrate to the target central tendency.

In [ ]:
# Build rating scale DataFrame
rs = df.groupby('grade_num').agg(
    n_obligors=('Creditability', 'count'),
    predicted_pd=('predicted_prob', 'mean'),
    observed_dr=('Creditability', 'mean'),
    score_min=('score', 'min'),
    score_max=('score', 'max')
).sort_index()

rs['pct_portfolio'] = rs['n_obligors'] / rs['n_obligors'].sum() * 100
rs['grade'] = [f"Grade {i}" for i in rs.index]

print("Rating Scale (pre-calibration):")
print(rs[['grade', 'score_min', 'score_max', 'predicted_pd', 'observed_dr', 'n_obligors', 'pct_portfolio']].to_string(index=False))

# Verify PD ordering: grades 1->N should have increasing PD (grade 1 = best)
pd_values = rs['predicted_pd'].values
is_monotonic = all(pd_values[i] <= pd_values[i+1] for i in range(len(pd_values)-1))
print(f"\nPD ordering strictly increasing: {is_monotonic}")

# Weighted average of predicted PDs (should approximate the sample mean predicted prob)
weighted_pred_pd = np.sum(rs['predicted_pd'] * rs['n_obligors']) / rs['n_obligors'].sum()
print(f"Weighted avg predicted PD: {weighted_pred_pd:.4f}")
print(f"Target CT: {TARGET_CT:.4f}")

In [ ]:
# Calibrate using model-predicted PDs (NOT observed DRs)
calib_result = pdt.rs_calibration(
    rs=rs.reset_index(),
    dr='predicted_pd',
    w='n_obligors',
    ct=TARGET_CT,
    min_pd=0.0003,
    method='scaling'
)

rs['calibrated_pd'] = calib_result.pd_calib
scaling_factor = calib_result.params.get('factor', None)

print(f"Calibration method: {calib_result.method}")
print(f"Scaling factor: {scaling_factor:.4f}")
print(f"\nCalibrated Rating Scale:")

# Calculate achieved CT
achieved_ct = np.sum(rs['calibrated_pd'] * rs['n_obligors']) / rs['n_obligors'].sum()
print(f"\nTarget CT: {TARGET_CT:.4f}")
print(f"Achieved CT: {achieved_ct:.4f}")
print(f"Difference: {abs(achieved_ct - TARGET_CT):.6f}")

display_cols = ['grade', 'score_min', 'score_max', 'predicted_pd', 'calibrated_pd', 'observed_dr', 'n_obligors', 'pct_portfolio']
print("\n" + rs[display_cols].to_string(index=False))

## 4. Self-Assessment (5 Checks)

In [ ]:
# Self-assessment checks
flags = []

# Check 1: Central tendency — |achieved - target| <= 0.005
ct_diff = abs(achieved_ct - TARGET_CT)
ct_pass = ct_diff <= 0.005
print(f"Check 1 — Central tendency: |{achieved_ct:.4f} - {TARGET_CT:.4f}| = {ct_diff:.6f} {'PASS' if ct_pass else 'FAIL'}")
if not ct_pass:
    flags.append(f"Central tendency deviation: {ct_diff:.4f} > 0.005")

# Check 2: Grade PD ordering — calibrated PDs strictly increasing
calib_pds = rs['calibrated_pd'].values
pd_ordering_valid = all(calib_pds[i] < calib_pds[i+1] for i in range(len(calib_pds)-1))
print(f"Check 2 — Grade PD ordering strictly increasing: {'PASS' if pd_ordering_valid else 'FAIL'}")
if not pd_ordering_valid:
    flags.append("Calibrated PDs not strictly increasing across grades")

# Check 3: Grade population — no grade < 2% or > 40%
pct_values = rs['pct_portfolio'].values
pop_min = pct_values.min()
pop_max = pct_values.max()
pop_pass = (pop_min >= 2.0) and (pop_max <= 40.0)
print(f"Check 3 — Grade population range: [{pop_min:.1f}%, {pop_max:.1f}%] {'PASS' if pop_pass else 'FAIL'}")
if not pop_pass:
    if pop_min < 2.0:
        flags.append(f"Grade with population < 2%: {pop_min:.1f}%")
    if pop_max > 40.0:
        flags.append(f"Grade with population > 40%: {pop_max:.1f}%")

# Check 4: Worst grade PD < 100%
worst_pd = calib_pds.max()
worst_pass = worst_pd < 1.0
print(f"Check 4 — Worst grade PD: {worst_pd:.4f} {'PASS' if worst_pass else 'FAIL'}")
if not worst_pass:
    flags.append(f"Worst grade PD >= 100%: {worst_pd:.4f}")

# Check 5: Grade concentration (HHI)
hhi_value = pdt.hhi(rs['n_obligors'].values)
hhi_pass = hhi_value < 0.25  # Moderate concentration threshold
print(f"Check 5 — Grade concentration HHI: {hhi_value:.4f} {'PASS' if hhi_pass else 'FAIL'}")
if not hhi_pass:
    flags.append(f"High grade concentration: HHI = {hhi_value:.4f}")

print(f"\n--- Self-Assessment Summary ---")
print(f"Checks passed: {sum([ct_pass, pd_ordering_valid, pop_pass, worst_pass, hhi_pass])}/5")
if flags:
    print(f"Flags: {flags}")
else:
    print("Flags: None")

## 5. Stress Testing

Apply PD stress shifts of +1% and +2% to the calibrated PDs and recalculate central tendency.

In [ ]:
# Stress testing: apply absolute PD shifts
stress_shifts = [0.01, 0.02]  # +1%, +2%
stress_results = {}

n_total = rs['n_obligors'].sum()

for shift in stress_shifts:
    stressed_pds = np.minimum(rs['calibrated_pd'].values + shift, 1.0)
    stressed_ct = np.sum(stressed_pds * rs['n_obligors'].values) / n_total
    stress_results[shift] = stressed_ct
    rs[f'stressed_pd_{int(shift*100)}pct'] = stressed_pds
    print(f"Stress +{shift*100:.0f}%: CT moves from {achieved_ct:.4f} to {stressed_ct:.4f} "
          f"(+{(stressed_ct - achieved_ct)*100:.2f} pp)")

print(f"\nStress test results:")
print(f"  Base CT:    {achieved_ct:.4f} ({achieved_ct*100:.2f}%)")
print(f"  +1% shift:  {stress_results[0.01]:.4f} ({stress_results[0.01]*100:.2f}%)")
print(f"  +2% shift:  {stress_results[0.02]:.4f} ({stress_results[0.02]*100:.2f}%)")

## 6. Plots

In [ ]:
# Plot 1: Rating Scale — Calibrated PDs (bars) vs Observed DRs (line)
fig, ax1 = plt.subplots(figsize=(10, 6))

grade_labels_plot = rs['grade'].values
x = np.arange(len(grade_labels_plot))
width = 0.5

bars = ax1.bar(x, rs['calibrated_pd'].values * 100, width, color='steelblue', alpha=0.8, label='Calibrated PD')
ax1.plot(x, rs['observed_dr'].values * 100, 'ro-', linewidth=2, markersize=8, label='Observed DR')
ax1.set_xlabel('Rating Grade', fontsize=12)
ax1.set_ylabel('PD / DR (%)', fontsize=12)
ax1.set_title('Rating Scale: Calibrated PDs vs Observed Default Rates', fontsize=14)
ax1.set_xticks(x)
ax1.set_xticklabels(grade_labels_plot, rotation=45, ha='right')
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# Add count labels on bars
for i, (bar, n) in enumerate(zip(bars, rs['n_obligors'].values)):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'n={n}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/05_rating_scale.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: figures/05_rating_scale.png")

In [ ]:
# Plot 2: Grade Distribution
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(x, rs['n_obligors'].values, width, color='teal', alpha=0.8)
ax.set_xlabel('Rating Grade', fontsize=12)
ax.set_ylabel('Number of Obligors', fontsize=12)
ax.set_title('Grade Distribution', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(grade_labels_plot, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Add percentage labels
for bar, pct in zip(bars, rs['pct_portfolio'].values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/05_grade_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: figures/05_grade_distribution.png")

In [ ]:
# Plot 3: Stress Test — Base vs Stressed PDs
fig, ax = plt.subplots(figsize=(10, 6))

bar_width = 0.25
x_pos = np.arange(len(grade_labels_plot))

ax.bar(x_pos - bar_width, rs['calibrated_pd'].values * 100, bar_width,
       color='steelblue', alpha=0.8, label='Base')
ax.bar(x_pos, rs['stressed_pd_1pct'].values * 100, bar_width,
       color='orange', alpha=0.8, label='Stress +1%')
ax.bar(x_pos + bar_width, rs['stressed_pd_2pct'].values * 100, bar_width,
       color='red', alpha=0.8, label='Stress +2%')

ax.set_xlabel('Rating Grade', fontsize=12)
ax.set_ylabel('PD (%)', fontsize=12)
ax.set_title('Stress Test: PD Shifts by Rating Grade', fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(grade_labels_plot, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/figures/05_stress_test.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: figures/05_stress_test.png")

## 7. Write Stage Summary and Fix Proposals

In [ ]:
# Build rating scale summary for stage_05.md
rs_summary_lines = []
for _, row in rs.iterrows():
    rs_summary_lines.append(
        f"  - grade: {row['grade']}\n"
        f"    score_range: [{row['score_min']:.0f}, {row['score_max']:.0f}]\n"
        f"    calibrated_pd: {row['calibrated_pd']:.6f}\n"
        f"    n_obligors: {int(row['n_obligors'])}\n"
        f"    pct_portfolio: {row['pct_portfolio']:.1f}"
    )

flags_str = str(flags) if flags else "None"

stage_05_md = f"""model_params_path: {RUN_DIR}/pipeline/model_params.json
target_central_tendency: {TARGET_CT:.4f}
achieved_central_tendency: {achieved_ct:.4f}
calibration_method: scaling
scaling_factor: {scaling_factor:.4f}
n_rating_grades: {len(rs)}
rating_scale:
{chr(10).join(rs_summary_lines)}
grade_pd_ordering_valid: {str(pd_ordering_valid).lower()}
hhi: {hhi_value:.4f}
stress_test:
  shift_1pct_ct: {stress_results[0.01]:.4f}
  shift_2pct_ct: {stress_results[0.02]:.4f}
calibration_flags: {flags_str}
"""

with open(f'{RUN_DIR}/pipeline/stage_05.md', 'w') as f:
    f.write(stage_05_md)
print(f"Written: {RUN_DIR}/pipeline/stage_05.md")

# Fix-proposer smoke tests
fix_issues = []

# Fix 1: Calibration not circular
n_grades_total = len(rs)
n_differ = sum(abs(rs['calibrated_pd'].values - rs['observed_dr'].values) > 0.01)
circular_pass = n_differ >= n_grades_total * 0.5
if not circular_pass:
    fix_issues.append(("CRITICAL", "Calibration may be circular: calibrated PD differs from observed DR by >0.01 for only "
                        f"{n_differ}/{n_grades_total} grades (need >= 50%)"))

# Fix 2: Scaling factor reasonable [0.5, 2.0]
sf_pass = 0.5 <= scaling_factor <= 2.0
if not sf_pass:
    fix_issues.append(("WARNING", f"Scaling factor {scaling_factor:.4f} outside reasonable range [0.5, 2.0]"))

# Fix 3: Central tendency achieved
ct_fix_pass = ct_diff <= 0.005
if not ct_fix_pass:
    fix_issues.append(("CRITICAL", f"Central tendency not achieved: |{achieved_ct:.4f} - {TARGET_CT:.4f}| = {ct_diff:.4f} > 0.005"))

# Fix 4: PD ordering
if not pd_ordering_valid:
    fix_issues.append(("CRITICAL", "Calibrated PDs not strictly monotonic across grades"))

# Fix 5: Grade population
min_pct = rs['pct_portfolio'].min()
if min_pct < 1.0:
    fix_issues.append(("WARNING", f"Grade with < 1% of portfolio: {min_pct:.1f}%"))

n_critical = sum(1 for sev, _ in fix_issues if sev == "CRITICAL")

fixes_md = f"""# Stage 05 Fix Proposals

## Smoke Tests
1. Calibration not circular: {"PASS" if circular_pass else "FAIL"} ({n_differ}/{n_grades_total} grades differ by >0.01)
2. Scaling factor reasonable: {"PASS" if sf_pass else "FAIL"} (factor={scaling_factor:.4f})
3. Central tendency achieved: {"PASS" if ct_fix_pass else "FAIL"} (diff={ct_diff:.6f})
4. PD ordering monotonic: {"PASS" if pd_ordering_valid else "FAIL"}
5. Grade population >= 1%: {"PASS" if min_pct >= 1.0 else "FAIL"} (min={min_pct:.1f}%)

## Issues Found: {len(fix_issues)} ({n_critical} critical)
"""

if fix_issues:
    for sev, desc in fix_issues:
        fixes_md += f"\n- [{sev}] {desc}"
else:
    fixes_md += "\nNo issues found."

with open(f'{RUN_DIR}/pipeline/stage_05_fixes.md', 'w') as f:
    f.write(fixes_md)
print(f"Written: {RUN_DIR}/pipeline/stage_05_fixes.md")

print(f"\n=== Stage 05 Summary ===")
print(f"Rating grades: {len(rs)}")
print(f"Target CT: {TARGET_CT*100:.2f}%, Achieved CT: {achieved_ct*100:.2f}%")
print(f"Grade PD ordering valid: {pd_ordering_valid}")
print(f"Scaling factor: {scaling_factor:.4f}")
print(f"Stress +1% CT: {stress_results[0.01]*100:.2f}%, Stress +2% CT: {stress_results[0.02]*100:.2f}%")
print(f"Flags: {flags_str}")
print(f"Fix proposals: {len(fix_issues)} issues ({n_critical} critical)")